## **1. Import Libraries and Configuration**

In [ ]:
!git clone https://github.com/jaeho0726/Sentiment-Analysis.git

%cd Sentiment-Analysis
!pip install -q -r requirements.txt

In [ ]:
!pip install -q konlpy

In [ ]:
import os
import json
import urllib.request

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("white")

import tqdm

from konlpy.tag import Okt

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.layers import Embedding, Dense, LSTM, Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from google.colab import drive, auth
from google.auth import default
import gspread

from src.preprocessing import clean_nsmc_data, prepare_journal_data

from src.sentiment import predict_konlpy_sentiment

from src.evaluation import evaluate_mood_alignment, get_largest_discrepancies, print_alignment_metrics

## **2. NSMC Training Data Preparation**

In [ ]:
drive.mount('/content/drive')

MODEL_PATH = '/content/drive/MyDrive/Project/Sentiment_Analysis/konlpy_model'

In [ ]:
TOKENIZER_PATH = os.path.join(MODEL_PATH, 'tokenizer.json')
CONFIG_PATH = os.path.join(MODEL_PATH, 'config.json')
KERAS_PATH = os.path.join(MODEL_PATH, 'lstm_model.keras')

MODEL_EXISTS = all([
    os.path.exists(TOKENIZER_PATH),
    os.path.exists(CONFIG_PATH),
    os.path.exists(KERAS_PATH)
])

if not MODEL_EXISTS:
  # Accessing Naver movie review data
  train_file = urllib.request.urlopen("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt")
  test_file = urllib.request.urlopen("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt")
  train_data = pd.read_table(train_file)
  test_data = pd.read_table(test_file)

  # Cleaning train data
  train_data = clean_nsmc_data(train_data)

  # Cleaning test data
  test_data = clean_nsmc_data(test_data)

  print("NSMC data loaded and cleaned")

else:
  print("Save model found - skipping NSMC data loading")

## **3. KoNLPy Model**

In [ ]:
okt = Okt()
stopwords = ['의', '가', '이', '은', '들', '는', '좀', '잘', '과', '도', '를', '으로', '자', '에', '와', '한', '하다']

TOKENIZER_PATH = os.path.join(MODEL_PATH, 'tokenizer.json')
CONFIG_PATH    = os.path.join(MODEL_PATH, 'config.json')
KERAS_PATH     = os.path.join(MODEL_PATH, 'lstm_model.keras')

if MODEL_EXISTS:
  # ── Load saved model, tokenizer, and config ──────────────────────────
  with open(TOKENIZER_PATH, 'r', encoding='utf-8') as f:
      tokenizer = tokenizer_from_json(f.read())

  with open(CONFIG_PATH, 'r') as f:
      config = json.load(f)
  vocab_size = config['vocab_size']
  max_len    = config['max_len']

  model = load_model(KERAS_PATH)
  print("Saved model loaded from", MODEL_PATH)

else:
  # ── Tokenization ─────────────────────────────────────────────────────
  X_train = []
  for sentence in tqdm.tqdm(train_data['document']):
      X_train.append([word for word in okt.morphs(sentence) if word not in stopwords])

  X_test = []
  for sentence in tqdm.tqdm(test_data['document']):
      X_test.append([word for word in okt.morphs(sentence) if word not in stopwords])

  # ── Vocabulary and rare word removal ─────────────────────────────────
  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(X_train)

  threshold  = 3
  rare_count = sum(1 for v in tokenizer.word_counts.values() if v < threshold)
  vocab_size = len(tokenizer.word_index) - rare_count + 2
  max_len    = 60

  tokenizer = Tokenizer(vocab_size, oov_token='OOV')
  tokenizer.fit_on_texts(X_train)
  X_train = tokenizer.texts_to_sequences(X_train)
  X_test  = tokenizer.texts_to_sequences(X_test)

  y_train = np.array(train_data['label'])
  y_test  = np.array(test_data['label'])

  # Drop empty sequences
  drop_train = [i for i, s in enumerate(X_train) if len(s) < 1]
  X_train = [s for i, s in enumerate(X_train) if i not in drop_train]
  y_train = np.delete(y_train, drop_train, axis=0)

  X_train = pad_sequences(X_train, maxlen=max_len)
  X_test  = pad_sequences(X_test,  maxlen=max_len)

  # ── Build and train model ─────────────────────────────────────────────
  model = Sequential()
  model.add(Embedding(vocab_size, 100, input_length=max_len))
  model.add(Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)))
  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
  model.summary()

  early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
  history = model.fit(X_train, y_train, epochs=10, batch_size=256,
                      validation_split=0.2, callbacks=[early_stop])
  test_loss, test_accuracy = model.evaluate(
  X_test,
  y_test,
  verbose=0)

  # ── Save model, tokenizer, and config ─────────────────────────────────
  os.makedirs(MODEL_PATH, exist_ok=True)

  model.save(KERAS_PATH)

  with open(TOKENIZER_PATH, 'w', encoding='utf-8') as f:
      f.write(tokenizer.to_json())

  with open(CONFIG_PATH, 'w') as f:
      json.dump({'vocab_size': vocab_size, 'max_len': max_len}, f)

  print("Training complete. Model saved to", MODEL_PATH)

## **4. Daily Reflection Sentiment Inference**

In [ ]:
auth.authenticate_user()

creds, _ = default()

gc = gspread.Client(auth=creds)

spreadsheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1DrDCLitpXZyJajcWYcMNcJCZU4Xa0aE_gj4sfXzl6x4/edit?gid=0#gid=0')

worksheet = spreadsheet.worksheet('data')

In [ ]:
df = pd.DataFrame(worksheet.get_all_records())

df = prepare_journal_data(df)

df[[
      "Date",
      "Today's Status",
      "Hours of Work",
      "Work Intensity",
      "Overall Mood",
      "Day_of_Week"
    ]
].head()

In [ ]:
df["Sentiment Score"] = df["Reflection"].apply(
    lambda reflection: predict_konlpy_sentiment(
        reflection=reflection,
        model=model,
        tokenizer=tokenizer,
        okt=okt,
        stopwords=stopwords,
        max_len=max_len
    )
)

df['Sentiment Score'] = pd.to_numeric(df['Sentiment Score'], errors='coerce')

df[[
      "Date",
      "Today's Status",
      "Hours of Work",
      "Work Intensity",
      "Overall Mood",
      "Day_of_Week",
      "Sentiment Score"
    ]
].head()

## **5. Validation Against Self-Reported Mood**

### **5.1 MAE / RMSE / Bias / Pearson Correlation**

In [ ]:
evaluation_df, metrics = evaluate_mood_alignment(df)

print_alignment_metrics(metrics)

### **5.2 Largest Discrepancies**

In [ ]:
largest_discrepancies = (
    get_largest_discrepancies(evaluation_df, n=10)
)

largest_discrepancies

### **5.3 Mood vs Sentiment Score Plot**

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    evaluation_df["Overall Mood"],
    evaluation_df["Sentiment Score"],
    alpha=0.7
)

plt.plot(
    [0, 10],
    [0, 10],
    linestyle="--",
    label="Perfect Agreement"
)

plt.xlim(0, 10)
plt.ylim(0, 10)

plt.xlabel("Self-Reported Overall Mood")
plt.ylabel("KoNLPy Sentiment Score")
plt.title("Self-Reported Mood vs KoNLPy Sentiment")

plt.legend()
plt.show()